In [70]:
# Modelo Dengue estocástico discreto (single strain) - 8 patches
import numpy as np


def _chk(name, x, low=None, high=None):
    if not np.all(np.isfinite(x)):
        i = np.where(~np.isfinite(x))[0]
        raise ValueError(f"{name}: NaN/Inf en índices {i[:10]}")
    if low is not None and np.any(x < low):
        i = np.where(x < low)[0]
        raise ValueError(f"{name}: valores < {low} en {i[:10]}, min={x.min()}")
    if high is not None and np.any(x > high):
        i = np.where(x > high)[0]
        raise ValueError(f"{name}: valores > {high} en {i[:10]}, max={x.max()}")

def a_from_temp(T):
    # ejemplo simple: mordeduras aumentan lineal hasta 32C, luego bajan
    a0 = 0.3
    return a0 * (1 + 0.05*(T - 25))  # simple; calibrar

def mu_v_from_temp(T):
    # mortalidad aumente fuera de óptimo
    base = 0.12
    return np.clip(base * (1 + 0.03*(T-25)), 0.05, 0.5)

def K_from_rain(rain, K0, alpha):
    # rain normalizado (asume rain en mm, normalizar por su max o por percentil fuera)
    return np.maximum(0.0, K0 * (1 + alpha * rain))

def simulate(
    days,
    Nh,            # array len P
    init_Ih,       # initial infected humans per patch
    temp, rain,    # arrays shape (P, days)
    adjacency,     # P x P movement probs or adjacency
    params,
    debug=False,
):
    P = len(Nh)
    # state arrays: shape (P,)
    Sh = Nh.copy().astype(int)
    Eh = np.zeros(P, dtype=int)
    Ih = np.array(init_Ih, dtype=int)
    Rh = np.zeros(P, dtype=int)

    # vector state
    Nv = (0.05 * Nh).astype(int)  # initial vector pop ~5% of humans; tune
    Sv = Nv.copy()
    Ev = np.zeros(P, dtype=int)
    Iv = np.zeros(P, dtype=int)






    # subtract initial infected humans from susceptible
    Sh -= Ih

    # time series store
    record = {'Sh':[], 'Eh':[], 'Ih':[], 'Rh':[], 'Sv':[], 'Ev':[], 'Iv':[], 'Nv':[]}

    # precompute movement matrix (row p: prob to go to q)
    # adjacency given as list of neighbors or matrix; here assume matrix P x P with rows summing to 1
    M = adjacency.copy()



    if debug:
        assert temp.shape == (P, days) and rain.shape == (P, days)
        assert M.shape == (P, P)
        _chk("Nh", Nh, 0, None)
        _chk("init_Ih", init_Ih, 0, None)
        _chk("M_row_sums", M.sum(1), 1.0, 1.0)

    for t in range(days):
        T = temp[:, t]
        R = rain[:, t]

        # compute climate-dependent params
        a = a_from_temp(T)             # per-patch biting rate
        a = np.clip(a, 0.0, 1.0)              # <-- evita a<0

        with np.errstate(divide='ignore', invalid='ignore'):
            lambda_h = a * params['b'] * (Iv / np.maximum(1, Nv))
            lambda_v = a * params['c'] * (Ih / np.maximum(1, Nh))

        lambda_h = np.clip(np.nan_to_num(lambda_h, nan=0.0), 0.0, None)  # <-- piso en 0
        lambda_v = np.clip(np.nan_to_num(lambda_v, nan=0.0), 0.0, None)
        if debug:
            _chk(f"lambda_h[t={t}]", lambda_h, 0, None)
            _chk(f"lambda_v[t={t}]", lambda_v, 0, None)

        mu_v = mu_v_from_temp(T)
        K0 = params['K0_base'] * (Nh/np.mean(Nh))  # scale K0 by patch population
        K = K_from_rain(R, K0, params['alpha_rain'])

        # update vector population via births (stochastic)
        births = np.random.poisson(np.maximum(0, K - Nv))
        Sv += births
        Nv += births

        # vector natural deaths
        deaths_v = np.random.binomial(Nv, 1 - np.exp(-mu_v))
        # remove deaths proportionally from Sv, Ev, Iv
        if Nv.sum() > 0:
            if deaths_v.sum() > 0:
                for p in range(P):
                    if Nv[p] == 0: continue
                    # split deaths into compartments proportionally
                    comps = np.array([Sv[p], Ev[p], Iv[p]], dtype=float)
                    if comps.sum() == 0:
                        Sv[p] = max(0, Sv[p] - deaths_v[p])
                    else:
                        fracs = comps / comps.sum()
                        dS = np.random.binomial(Sv[p], (deaths_v[p]*fracs[0]/max(1e-9,comps[0])))
                        # simpler: proportional remove
                        Sv[p] = max(0, Sv[p] - int(round(deaths_v[p]*fracs[0])))
                        Ev[p] = max(0, Ev[p] - int(round(deaths_v[p]*fracs[1])))
                        Iv[p] = max(0, Iv[p] - int(round(deaths_v[p]*fracs[2])))
        Nv = Sv + Ev + Iv  # refresh

        # Forces of infection
        with np.errstate(divide='ignore', invalid='ignore'):
            lambda_h = a * params['b'] * (Iv / np.maximum(1, Nv))
            lambda_v = a * params['c'] * (Ih / np.maximum(1, Nh))

        # HUMAN MOVEMENT (move people according to M)
        # For each compartment, redistribute across patches using multinomial draws
        def move_compartment(X):
            X_new = np.zeros_like(X)
            for p in range(P):
                n = X[p]
                if n <= 0: continue
                probs = M[p]
                # draw multinomial: how many from p go to each q
                moved = np.random.multinomial(n, probs)
                X_new += moved
            return X_new

        Sh = move_compartment(Sh)
        Eh = move_compartment(Eh)
        Ih = move_compartment(Ih)
        Rh = move_compartment(Rh)

        # HUMAN transitions (exposure, progression, recovery)
        newE = np.array([np.random.binomial(Sh[p], 1 - np.exp(-lambda_h[p])) for p in range(P)])
        Sh -= newE
        Eh += newE

        newI = np.array([np.random.binomial(Eh[p], 1 - np.exp(-params['sigma_h'])) for p in range(P)])
        Eh -= newI
        Ih += newI

        newR = np.array([np.random.binomial(Ih[p], 1 - np.exp(-params['gamma'])) for p in range(P)])
        Ih -= newR
        Rh += newR

        record.setdefault('newIh', []).append(newI.copy())
        # VECTOR transitions: infections (from humans), progression E->I
        newEv = np.array([np.random.binomial(Sv[p], 1 - np.exp(-lambda_v[p])) for p in range(P)])
        Sv -= newEv
        Ev += newEv

        newIv = np.array([np.random.binomial(Ev[p], 1 - np.exp(-params['sigma_v'])) for p in range(P)])
        Ev -= newIv
        Iv += newIv

        # store
        record['Sh'].append(Sh.copy()); record['Eh'].append(Eh.copy())
        record['Ih'].append(Ih.copy()); record['Rh'].append(Rh.copy())
        record['Sv'].append(Sv.copy()); record['Ev'].append(Ev.copy())
        record['Iv'].append(Iv.copy()); record['Nv'].append(Nv.copy())

    # convert lists to arrays
    for k in record:
        record[k] = np.array(record[k])
    return record




In [71]:
import pandas as pd


def infecciosos_df(casos, pre=2, post=5, group_col='municerca'):
    c = casos.assign(
        start = casos['fis'].dt.normalize() - pd.to_timedelta(pre, 'D'),
        end   = casos['fis'].dt.normalize() + pd.to_timedelta(post, 'D') + pd.Timedelta(days=1)  # fin exclusivo
    )[[group_col, 'start', 'end']]

    # eventos (+1 al inicio, -1 al fin_exclusivo)
    ev_start = c.rename(columns={'start':'fecha'})[[group_col, 'fecha']]
    ev_start['delta'] = 1
    ev_end   = c.rename(columns={'end':'fecha'})[[group_col, 'fecha']]
    ev_end['delta'] = -1
    ev = pd.concat([ev_start, ev_end], ignore_index=True)

    # tabla wide de deltas por día y grupo
    wide_delta = (ev.pivot_table(index='fecha', columns=group_col, values='delta', aggfunc='sum')
                    .sort_index()
                    .fillna(0))

    # rango completo y ceros
    idx = pd.date_range(wide_delta.index.min(), wide_delta.index.max(), freq='D')
    wide_delta = wide_delta.reindex(idx, fill_value=0)

    # cumsum por columna -> infecciosos por fecha y grupo (incluye días sin casos)
    wide = wide_delta.cumsum()
    wide.index.name = 'fecha'
    return wide.astype('int32')

import pandas as pd
df_cases=pd.read_csv(r"data\processed\diario_municerca.csv").sort_values("fis")
df_cases=df_cases.loc[~df_cases["municerca"].isnull()]
df_cases['fis'] = pd.to_datetime(df_cases['fis'], errors='coerce')

# dedup + ventanas
casos = (df_cases.drop_duplicates('ideventocaso')
           .assign(fis=pd.to_datetime(df_cases['fis'], errors='coerce'))
           .dropna(subset=['fis']))

wide = infecciosos_df(casos, pre=2, post=5, group_col='municerca')   # DataFrame ancho
# total sistema
wide['TOTAL'] = wide.sum(axis=1)

# formato largo (opcional)
long = (wide.drop(columns=['TOTAL'])
            .stack()
            .rename('infecciosos')
            .reset_index())
f_inicial="2024-01-20"
long=long.sort_values(by="municerca")
long_f=long.loc[long["fecha"]==f_inicial]

In [ ]:


df=pd.read_excel(r"data\external\Indicadores sociodemograficos\Indicadores_MundoSano.xlsx").sort_values(by="municerca")

df2=df[["municerca","Estimación poblacional"]]

Nh=df2['Estimación poblacional'].to_numpy(dtype=int)   # -> array([...])

init_Ih=long_f['infecciosos'].to_numpy(dtype=int)   # -> array([...])


array([0, 0, 1, 1, 1, 2, 1, 4])

In [73]:


df_clima=pd.read_csv("data\processed\clima_processed.csv")
df_clima
import pandas as pd
import numpy as np

def build_matrix(df, value_col, start_date, days, patch_col='municerca', date_col='date'):
    # fechas objetivo
    idx = pd.date_range(pd.to_datetime(start_date).normalize(), periods=days, freq='D')

    # tipos
    g = df.copy()
    g[date_col] = pd.to_datetime(g[date_col]).dt.normalize()

    # elegí el conjunto y orden de patches
    patches = (g[patch_col].dropna().unique().tolist())   # o tu orden preferido

    # filtro por rango y pivoteo -> (days x P)
    tbl = (g.loc[g[date_col].between(idx.min(), idx.max())]
             .pivot_table(index=date_col, columns=patch_col, values=value_col, aggfunc='mean')
             .reindex(idx))                 # asegura todas las fechas

    # completa faltantes (interpolar por columna y ffill/bfill de respaldo)
    tbl = tbl.interpolate(limit_direction='both').ffill().bfill()

    # reordená/seleccioná columnas y exportá
    tbl = tbl.reindex(columns=patches)
    arr = tbl.to_numpy().T.astype(float)    # -> (P, days)
    return arr, patches, idx

# ejemplo:
start_date = '2024-01-20'
days = 180

temp, patches, dates = build_matrix(df_clima, 't2m', start_date, days)  # (P, days)
rain, _, _            = build_matrix(df_clima, 'tp',  start_date, days)  # idem

P = temp.shape[0]
assert temp.shape == (P, days) and rain.shape == (P, days)


<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Nainh\AppData\Local\Temp\ipykernel_12580\3760624634.py:1: SyntaxWarning: invalid escape sequence '\p'
  df_clima=pd.read_csv("data\processed\clima_processed.csv")


In [74]:
# ===== Example setup =====
P = 8
days = 180
# Nh = np.array([20000]*P)  # ejemplo: cada patch 20k habitantes; reemplazar con datos reales
# init_Ih = np.zeros(P, int); init_Ih[0] = 5  # semilla en patch 0
# temp = 25 + 3*np.random.randn(P, days)  # ejemplo sintético
# rain = np.abs(np.random.randn(P, days))  # mm normalizado; reemplazar con series reales

# adjacency: ejemplo simple de línea con movimiento a vecinos y quedarse
M = np.zeros((P,P))
for p in range(P):
    neighbors = []
    if p-1 >= 0: neighbors.append(p-1)
    neighbors.append(p)
    if p+1 < P: neighbors.append(p+1)
    # distribuir probabilidades
    probs = np.zeros(P)
    probs[neighbors] = 1.0/len(neighbors)
    M[p] = probs

base_params = {
    'b': 0.4,
    'c': 0.5,
    'sigma_h': 1/5.0,    # incubation human
    'gamma': 1/5.0,      # recovery
    'sigma_v': 1/10.0,   # EIP
    'K0_base': 500.0,    # base carrying capacity scale (tune)
    'alpha_rain': 0.6
}

#rec = simulate(days, Nh, init_Ih, temp, rain, M, base_params,debug=True)
# rec['Ih'] shape (days, P)


In [88]:
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from scipy.special import gammaln

# ---------- 1) Observado semanal por FIS (weeks x P) ----------
def obs_weekly(df_cases, order, start_date, days, week='W-SUN'):
    d = (df_cases.drop_duplicates('ideventocaso')
                 .assign(fis=pd.to_datetime(df_cases['fis']).dt.normalize()))
    idx = pd.date_range(pd.to_datetime(start_date).normalize(), periods=days, freq='D')
    g = (d[d['fis'].between(idx[0], idx[-1])]
           .groupby(['fis','municerca'])['ideventocaso'].nunique()
           .unstack('municerca')
           .reindex(idx).fillna(0.0))
    w = g.resample(week).sum()
    y = w.reindex(columns=order).to_numpy()  # (weeks, P)
    return y, w.index

# ---------- 2) Incidencia semanal simulada promedio (weeks x P) ----------
def sim_weekly(theta, R=5):
    b, c, K0_base, alpha = theta
    params = base_params.copy()
    params.update(b=b, c=c, K0_base=K0_base, alpha_rain=alpha)

    runs = []
    for _ in range(R):
        rec = simulate(days, Nh, init_Ih, temp, rain, M, params)
        Ih, Rh = rec['Ih'], rec['Rh']                # (days, P)
        inc = np.diff(Ih, axis=0) + np.diff(Rh, axis=0)   # (days-1, P)
        inc = np.vstack([np.zeros((1, inc.shape[1])), inc])  # (days, P) alineado a días
        W = (inc.shape[0] // 7)
        inc_w = inc[:W*7].reshape(W, 7, inc.shape[1]).sum(axis=1)  # (weeks, P)
        runs.append(inc_w)
    lam = np.mean(runs, axis=0) + 1e-6  # media como intensidad Poisson
    return lam

# ---------- 3) Función objetivo: NLL Poisson ----------
def poisson_nll(lam, y):
    # lam,y: (weeks, P)
    return np.sum(lam - y*np.log(lam) + gammaln(y+1))

def make_objective(y):
    def obj(theta):
        lam = sim_weekly(theta, R=5)
        W = min(lam.shape[0], y.shape[0])
        return poisson_nll(lam[:W], y[:W])
    return obj



order=list(df_cases.sort_values("municerca")["municerca"].unique())
# ---------- 4) Correr la calibración ----------
# Prepará INSUMOS: Nh, init_Ih, temp, rain, M, base_params, df_cases, order, start_date, days
# - order: orden de parches (lista de nombres) p.ej. list(pop_df['municerca'])
y_obs, week_index = obs_weekly(df_cases, order, start_date, days)
objective = make_objective(y_obs)

# bounds = [
#     (0.05, 0.6),   # b
#     (0.2,  0.8),   # c
#     (100.0, 20000.0),  # K0_base
#     (0.0,  2.0),   # alpha_rain
# ]

# res = differential_evolution(objective, bounds, popsize=12, maxiter=40, tol=1e-2, seed=0)
# best = dict(b=res.x[0], c=res.x[1], K0_base=res.x[2], alpha_rain=res.x[3], nll=res.fun)
# print(best)


In [90]:
import numpy as np
from scipy.optimize import differential_evolution
from scipy.special import gammaln

base_params = dict(
    b=0.3, c=0.5, sigma_h=1/5, gamma=1/5, sigma_v=1/10, K0_base=500.0, alpha_rain=0.6
)

def weekly_from_rec(rec):
    Ih, Rh = rec['Ih'], rec['Rh']            # (days, P)
    inc = np.diff(Ih, axis=0) + np.diff(Rh, axis=0)  # (days-1, P)
    inc = np.vstack([np.zeros((1, inc.shape[1])), inc])  # alinear a días
    W = inc.shape[0] // 7
    return inc[:W*7].reshape(W, 7, inc.shape[1]).sum(axis=1)  # (weeks, P)

def sim_weekly(theta, R=3):
    b, c, K0_base, alpha = theta
    params = base_params.copy()
    params.update(b=float(b), c=float(c), K0_base=float(K0_base), alpha_rain=float(alpha))

    runs = []
    for _ in range(R):
        rec = simulate(days, Nh, init_Ih, temp, rain, M, params)
        runs.append(weekly_from_rec(rec))
    lam = np.mean(runs, axis=0)
    # seguridad numérica
    lam = np.clip(lam, 1e-9, None)
    return lam

def poisson_nll(lam, y):
    return np.sum(lam - y*np.log(lam) + gammaln(y+1))

def make_objective(y):
    def obj(theta):
        try:
            lam = sim_weekly(theta, R=3)
            W = min(lam.shape[0], y.shape[0])
            return float(poisson_nll(lam[:W], y[:W]))
        except Exception as e:
            # Si algo explota (shapes, NaNs, etc.), penalizá fuerte
            return 1e15
    return obj
objective = make_objective(y_obs)

# sanity check:
val = objective([0.3, 0.5, 500.0, 0.6])
print("obj @ start:", val)   # debe ser número finito

obj @ start: 43442.37082691094


In [ ]:
bounds = [
    (0.05, 0.6),      # b
    (0.2,  0.8),      # c
    (100.0, 20000.0), # K0_base
    (0.0,  2.0),      # alpha_rain
]
res = differential_evolution(objective, bounds, popsize=12, maxiter=40, tol=1e-2, seed=0)  # sin workers
print(res.x, res.fun) #array([5.19052844e-01, 4.30579009e-01, 2.75900123e+02, 2.26056206e-04])

[5.19052844e-01 4.30579009e-01 2.75900123e+02 2.26056206e-04] 3181.4887213353477


In [101]:
res.x

array([5.19052844e-01, 4.30579009e-01, 2.75900123e+02, 2.26056206e-04])

In [92]:
import numpy as np
from scipy.special import gammaln

# --- helpers ---
def align(y, lam):
    W = min(y.shape[0], lam.shape[0])
    return y[:W], lam[:W]

def poisson_nll(lam, y):
    lam = np.clip(lam, 1e-9, None)
    return float(np.sum(lam - y*np.log(lam) + gammaln(y+1)))

def rmse_overall(lam, y):
    # RMSE sobre todas las celdas (weeks×P)
    return float(np.sqrt(np.mean((lam - y)**2)))

def rmse_by_patch(lam, y):
    # vector de RMSE por parche
    return np.sqrt(np.mean((lam - y)**2, axis=0))

def rmse_weekly_totals(lam, y):
    # RMSE de totales semanales agregados (suma sobre parches)
    return float(np.sqrt(np.mean((lam.sum(axis=1) - y.sum(axis=1))**2)))

# --- predicciones promedio para reducir ruido estocástico ---
best_theta = res.x
base_theta = [base_params['b'], base_params['c'],
              base_params['K0_base'], base_params['alpha_rain']]

lam_best = sim_weekly(best_theta, R=10)   # (weeks, P)
lam_base = sim_weekly(base_theta, R=10)

y, lam_best = align(y_obs, lam_best)
_, lam_base = align(y_obs, lam_base)

# --- métricas ---
nll_best = poisson_nll(lam_best, y)
nll_base = poisson_nll(lam_base, y)

rmse_best = rmse_overall(lam_best, y)
rmse_base = rmse_overall(lam_base, y)

rmse_best_by_patch = rmse_by_patch(lam_best, y)
rmse_base_by_patch = rmse_by_patch(lam_base, y)

rmse_best_tot = rmse_weekly_totals(lam_best, y)
rmse_base_tot = rmse_weekly_totals(lam_base, y)

print("Poisson NLL  best:", nll_best)
print("Poisson NLL  base:", nll_base)
print("RMSE overall best:", rmse_best)
print("RMSE overall base:", rmse_base)
print("RMSE totals  best:", rmse_best_tot)
print("RMSE totals  base:", rmse_base_tot)
print("RMSE por parche (best):", rmse_best_by_patch)
print("RMSE por parche (base):", rmse_base_by_patch)
def make_objective_rmse(y):
    def obj(theta):
        try:
            lam = sim_weekly(theta, R=3)
            y_, lam_ = align(y, lam)
            return rmse_overall(lam_, y_)
        except Exception:
            return 1e15
    return obj

objective_rmse = make_objective_rmse(y_obs)

# res_rmse = differential_evolution(objective_rmse, bounds, popsize=12, maxiter=40, tol=1e-2, seed=0)  # sin workers
# res_rmse.x

Poisson NLL  best: 3981.8259834365317
Poisson NLL  base: 37352.01621823175
RMSE overall best: 28.83249815745698
RMSE overall base: 32.704784817926715
RMSE totals  best: 214.15506438094582
RMSE totals  base: 249.61253734365752
RMSE por parche (best): [25.59191279 16.30759332 32.16818304 13.37869949 24.5571741  35.32305196
 37.44336523 35.53955543]
RMSE por parche (base): [28.0767377  19.18185601 36.74501327 16.97723181 29.47349318 39.74447383
 42.49248404 38.83597816]


In [93]:
import pandas as pd
import numpy as np

def obs_weekly(df_cases, order, start_date, days, week_freq='W-SUN'):
    # normaliza fechas y dedup casos
    d = (df_cases.drop_duplicates('ideventocaso')
                  .assign(fis=pd.to_datetime(df_cases['fis']).dt.normalize()))
    # índice diario objetivo
    idx = pd.date_range(pd.to_datetime(start_date).normalize(), periods=days, freq='D')
    # panel diario (days × P) y resample a semanas
    daily = (d[d['fis'].between(idx[0], idx[-1])]
               .groupby(['fis','municerca'])['ideventocaso'].nunique()
               .unstack('municerca')
               .reindex(idx)
               .fillna(0.0)
               .reindex(columns=order, fill_value=0.0))
    weekly = daily.resample(week_freq).sum()              # (W × P)
    return weekly.to_numpy(), weekly.index                # y_obs, week_index
def seir_weekly_from_rec(rec):
    # rec['Ih'], rec['Rh']: (days × P)
    Ih, Rh = rec['Ih'], rec['Rh']
    inc = np.diff(Ih, axis=0) + np.diff(Rh, axis=0)      # (days-1 × P)
    inc = np.vstack([np.zeros((1, inc.shape[1])), inc])  # alinear a días: (days × P)
    W = inc.shape[0] // 7
    return inc[:W*7].reshape(W, 7, inc.shape[1]).sum(axis=1)  # (weeks × P)

def seir_weekly_from_rec(rec, key='newIh'):
    inc = np.asarray(rec[key])            # (days, P)  no-negativo
    W = inc.shape[0] // 7
    return inc[:W*7].reshape(W, 7, inc.shape[1]).sum(axis=1)  # (weeks, P)


In [ ]:
# insumos que ya tenés
# order = list(pop_df['municerca'])     # orden de parches (columnas)
start_date = '2024-01-01'
days = 180

# observado
y_obs, week_index = obs_weekly(df_cases, order, start_date, days)


params={'b': float(res.x[0]),
 'c': float(res.x[1]),
 'sigma_h': 0.2,
 'gamma': 0.2,
 'sigma_v': 0.1,
 'K0_base': float(res.x[2]),
 'alpha_rain': float(res.x[3]),}

# SEIR (promediado si querés reducir ruido)
R = 10
seir_runs = []
for _ in range(10):
    rec = simulate(days, Nh, init_Ih, temp, rain, M, base_params)
    seir_runs.append(seir_weekly_from_rec(rec, 'newIh'))
lam_seir = np.mean(seir_runs, axis=0)
W = min(y_obs.shape[0], lam_seir.shape[0])
y_obs   = y_obs[:W]
lam_seir= lam_seir[:W]


array([[0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0.1, 0. , 0.1, 0. , 0.1],
       [0. , 0. , 0. , 0.1, 0.1, 0. , 0. , 0. ],
       [0. , 0. , 0.2, 0. , 0.1, 0.1, 0. , 0. ],
       [0. , 0. , 0. , 0.1, 0.1, 0.1, 0.1, 0. ],
       [0. , 0. , 0.1, 0.2, 0.1, 0.1, 0.1, 0. ],
       [0. , 0. , 0. , 0.1, 0. , 0.2, 0. , 0. ],
       [0. , 0. , 0.2, 0.5, 0. , 0.2, 0.1, 0. ],
       [0. , 0.1, 0.1, 0.1, 0.1, 0.3, 0.1, 0.2],
       [0. , 0.1, 0. , 0. , 0. , 0.1, 0. , 0. ],
       [0. , 0.1, 0.1, 0. , 0.3, 0. , 0.1, 0.2],
       [0. , 0.1, 0.1, 0.2, 0.3, 0. , 0.1, 0. ],
       [0. , 0. , 0.5, 0.3, 0.1, 0.1, 0.2, 0. ],
       [0. , 0.1, 0.4, 0. , 0.1, 0.1, 0. , 0.1],
       [0. , 0. , 0.1, 0.4, 0.1, 0.3, 0.3, 0. ],
       [0.1, 0.3, 0.3, 0.3, 0.1, 0. , 0.5, 0.1],
       [0.1, 0.4, 0.4, 0.9, 0.5, 0.5, 0.4, 0. ],
       [0.3, 1. , 1.

In [95]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_poisson_deviance

def metrics_block(y, yhat):
    W = min(y.shape[0], yhat.shape[0])
    y, yhat = y[:W], yhat[:W]
    tot_y, tot_hat = y.sum(axis=1), yhat.sum(axis=1)  # totales por semana

    m = {
        "RMSE_all":  float(np.sqrt(mean_squared_error(y.ravel(), yhat.ravel()))),
        "MAE_all":   float(mean_absolute_error(y.ravel(), yhat.ravel())),
        "PoissDev_all": float(mean_poisson_deviance(y.ravel(), np.clip(yhat.ravel(), 1e-9, None))),
        "RMSE_tot":  float(np.sqrt(mean_squared_error(tot_y, tot_hat))),
        "MAE_tot":   float(mean_absolute_error(tot_y, tot_hat)),
        "PoissDev_tot": float(mean_poisson_deviance(tot_y, np.clip(tot_hat, 1e-9, None))),
    }
    # RMSE por parche
    rmse_p = np.sqrt(((y - yhat)**2).mean(axis=0))
    m.update({f"RMSE_p{j}": float(v) for j, v in enumerate(rmse_p)})
    return m

# ------- elegí el bloque de test -------
# Ejemplo: últimas 8 semanas como test
H = 8
W = min(y_obs.shape[0], lam_seir.shape[0])
t0 = W - H

y_test    = y_obs[t0:W]
seir_hat  = lam_seir[t0:W]
weeks_ts  = week_index[t0:W]  # por si querés etiquetar/plotear

# métricas SEIR
m_seir = metrics_block(y_test, seir_hat)
print("SEIR:", m_seir)

SEIR: {'RMSE_all': 3.875907151880705, 'MAE_all': 2.4046874999999996, 'PoissDev_all': 18.418538395296373, 'RMSE_tot': 28.932097227819483, 'MAE_tot': 19.237499999999997, 'PoissDev_tot': 63.07969081747571, 'RMSE_p0': 2.9404506457344257, 'RMSE_p1': 2.071231517720798, 'RMSE_p2': 3.989360851063739, 'RMSE_p3': 3.1788362650504665, 'RMSE_p4': 3.894226495724151, 'RMSE_p5': 3.6245689398878866, 'RMSE_p6': 4.2262572093993525, 'RMSE_p7': 5.921254090140028}


In [96]:
print("y_obs shape:", y_obs.shape)          # (weeks, P?)
print("lam_seir shape:", lam_seir.shape)    # (weeks, P?)
print("P_obs,P_seir:", y_obs.shape[1], lam_seir.shape[1])


y_obs shape: (25, 8)
lam_seir shape: (25, 8)
P_obs,P_seir: 8 8
